# Train Brick Transformer (Colab GPU)
Decoder-only Transformer. Fast config for T4 16GB.
- Causal mask in __init__ (once)
- Mixed precision FP16 (torch.amp)
- Loss uses ALL positions
- ~20 min total for 5 epochs

In [ ]:
import os, sys, json, math, time, gc
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {mem:.1f} GB')

In [ ]:
# @title Settings
POSSIBLE_PATHS = ['/content/training_data/projects.json', '/content/aip/training_data/projects.json']
PROJECTS_JSON = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)
MODEL_DIR = '/content/model'
os.makedirs(MODEL_DIR, exist_ok=True)

WINDOW = 256
EPOCHS = 5
BATCH_SIZE = 8
LR = 3e-4
D_MODEL = 128
N_LAYERS = 4
N_HEADS = 4
D_FF = 512
DROPOUT = 0.1
USE_COMPILE = False

In [ ]:
# @title Upload projects.json
from google.colab import files
if PROJECTS_JSON is None or not os.path.exists(PROJECTS_JSON):
    print('Upload projects.json:')
    uploaded = files.upload()
    PROJECTS_JSON = '/content/' + next(iter(uploaded.keys()))

with open(PROJECTS_JSON, 'r', encoding='utf-8') as f:
    projects = json.load(f)
num_projects = len(projects)
print(f'Projects: {num_projects}')

In [ ]:
# @title Build vocabulary
SPECIAL = ['[PAD]','[UNK]','[SEP]','[START]','[END]']
STRUCTURAL = ['<project_start>','<project_end>','<scene_start>','<scene_end>',
              '<object_start>','<object_end>','<script_start>','<script_end>',
              '<global_var>','<global_list>','<broadcast>','<signal>']
PAD_ID, UNK_ID = 0, 1

counter = Counter()
for proj in projects:
    for scene in proj.get('scenes', []):
        for sprite in scene.get('sprites', []):
            for script in sprite.get('scripts', []):
                for brick in script.get('bricks', []):
                    bt = brick.get('type', '')
                    if bt: counter[bt] += 1

word2id = {t: i for i, t in enumerate(SPECIAL)}
for t in STRUCTURAL: word2id[t] = len(word2id)
for w, _ in counter.most_common():
    if w not in word2id: word2id[w] = len(word2id)
id2word = {v: k for k, v in word2id.items()}
vocab_size = len(word2id)
print(f'Vocab: {vocab_size}')

In [ ]:
# @title Build dataset
def encode(bt): return word2id.get(bt, UNK_ID)

def build_seq(proj, max_len=3000):
    t = [encode('<project_start>')]
    for scene in proj.get('scenes', []):
        t.append(encode('<scene_start>'))
        for sprite in scene.get('sprites', []):
            t.append(encode('<object_start>'))
            for script in sprite.get('scripts', []):
                t.append(encode('<script_start>'))
                for brick in script.get('bricks', []):
                    bt = brick.get('type', '')
                    if bt: t.append(encode(bt))
                t.append(encode('<script_end>'))
            t.append(encode('<object_end>'))
        t.append(encode('<scene_end>'))
    t.append(encode('<project_end>'))
    return t[-max_len:]

class BrickDataset(Dataset):
    def __init__(self, projects, window=256):
        self.inputs, self.targets = [], []
        for proj in projects:
            seq = build_seq(proj, window * 3)
            if len(seq) <= window: continue
            for i in range(window, len(seq)):
                ctx = seq[i-window:i]
                tgt = seq[i]
                if tgt in (PAD_ID, 3): continue
                self.inputs.append(ctx)
                self.targets.append(seq[i-window+1 : i+1])
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

print('Building dataset...')
t0 = time.time()
dataset = BrickDataset(projects, window=WINDOW)
del projects; gc.collect()
n_pairs = len(dataset)
print(f'Pairs: {n_pairs} ({time.time()-t0:.0f}s)')

In [ ]:
# @title DataLoader
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
print(f'Batches: {len(dataloader)}')

In [ ]:
# @title Transformer
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1), :]

class DecBlock(nn.Module):
    def __init__(self, d, h, ff, drop):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, dropout=drop, batch_first=True)
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d,ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff,d), nn.Dropout(drop))
        self.do = nn.Dropout(drop)
    def forward(self, x, m):
        a, _ = self.attn(x, x, x, attn_mask=m)
        x = self.n1(x + self.do(a))
        return self.n2(x + self.ff(x))

class BrickTransformer(nn.Module):
    def __init__(self, vs, d=128, nl=4, nh=4, ff=512, drop=0.1, max_len=256):
        super().__init__()
        self.tok_emb = nn.Embedding(vs, d, padding_idx=0)
        self.pos_enc = PositionalEncoding(d, max_len)
        self.do = nn.Dropout(drop)
        self.blocks = nn.ModuleList([DecBlock(d, nh, ff, drop) for _ in range(nl)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vs)
        self.register_buffer('mask', torch.triu(torch.full((max_len, max_len), float('-inf')), diagonal=1))
        for p in self.parameters():
            if p.dim() > 1: nn.init.normal_(p, 0, 0.02)
    def forward(self, x):
        m = self.mask[:x.size(1), :x.size(1)]
        x = self.tok_emb(x)
        x = self.pos_enc(x)
        x = self.do(x)
        for b in self.blocks: x = b(x, m)
        return self.head(self.ln(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BrickTransformer(vocab_size, D_MODEL, N_LAYERS, N_HEADS, D_FF, DROPOUT, WINDOW).to(device)
_model_raw = model
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {n_params:,}')

In [ ]:
# @title Training (~20 min)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
total_steps = len(dataloader) * EPOCHS
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=total_steps, pct_start=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=(torch.cuda.is_available()))

print(f'Training: {EPOCHS} epochs x {len(dataloader)} batches')
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'Model on: {next(model.parameters()).device}')
t0 = time.time()
best_loss = float('inf')
for epoch in range(EPOCHS):
    model.train()
    tl = tc = tt = 0.0
    for bi, (x, y) in enumerate(dataloader):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=0)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        valid = y != 0
        tl += loss.item() * valid.sum().item()
        preds = logits.argmax(-1)
        tc += (preds[valid] == y[valid]).sum().item()
        tt += valid.sum().item()
        if bi % 2000 == 0 and bi > 0:
            print(f'  B{bi}/{len(dataloader)} loss={tl/max(tt,1):.4f} | {time.time()-t0:.0f}s')
    loss = tl / max(tt, 1)
    acc = tc / max(tt, 1)
    print(f'Epoch {epoch+1}/{EPOCHS} | loss={loss:.4f} acc={acc:.4f} | {time.time()-t0:.0f}s | lr={scheduler.get_last_lr()[0]:.2e}')
    if loss < best_loss:
        best_loss = loss
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'transformer_model.pt'))
print(f'Done: {time.time()-t0:.0f}s')

In [ ]:
# @title Save + ONNX
!pip install -q onnx onnxscript

model_path = os.path.join(MODEL_DIR, 'transformer_model.pt')
torch.save(model.state_dict(), model_path)
print(f'PyTorch: {os.path.getsize(model_path)/1024/1024:.2f} MB')

try:
    model.eval()
    class OnnxWrapper(nn.Module):
        def __init__(self, m):
            super().__init__()
            self.inner = m
        def forward(self, x):
            return self.inner(x.to(torch.long))[:, -1, :]
    wrapper = OnnxWrapper(_model_raw)
    example = torch.zeros((1, WINDOW), dtype=torch.float32, device=device)
    onnx_path = os.path.join(MODEL_DIR, 'transformer_model.onnx')
    torch.onnx.export(wrapper, example, onnx_path,
        input_names=['input_ids'], output_names=['logits'],
        opset_version=14, dynamo=False)
    print(f'ONNX: {os.path.getsize(onnx_path)/1024/1024:.2f} MB')
except Exception as e: print(f'ONNX: {e}')

vocab_data = {'word2id': word2id, 'id2word': {str(k):v for k,v in id2word.items()}, 'vocab_size': vocab_size, 'structural_tokens': STRUCTURAL}
with open(os.path.join(MODEL_DIR, 'vocab.json'), 'w', encoding='utf-8') as f: json.dump(vocab_data, f, indent=2, ensure_ascii=False)

metadata = {'model_version': 4, 'model_type': 'transformer', 'vocab_size': vocab_size, 'window': WINDOW, 'brick_types': list(word2id.keys())}
with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w') as f: json.dump(metadata, f, indent=2)

stats = {'model_type':'transformer','vocab_size':vocab_size,'window':WINDOW,'d_model':D_MODEL,'n_layers':N_LAYERS,'n_heads':N_HEADS,'d_ff':D_FF,'n_params':n_params,'pairs':n_pairs,'epochs':EPOCHS,'batch_size':BATCH_SIZE}
try: stats['best_loss'] = round(best_loss,4)
except: pass
try: stats['train_time'] = round(train_time,1)
except: pass
try: stats['final_acc'] = round(acc,4)
except: pass
with open(os.path.join(MODEL_DIR, 'training_stats.json'), 'w') as f: json.dump(stats, f, indent=2)

print('DONE')
print(model_path)
print(onnx_path)

In [ ]:
# @title Download
from google.colab import files
files.download(model_path)
if os.path.exists(onnx_path): files.download(onnx_path)
files.download(os.path.join(MODEL_DIR, 'vocab.json'))
files.download(os.path.join(MODEL_DIR, 'training_stats.json'))
files.download(os.path.join(MODEL_DIR, 'model_metadata.json'))